# RG3 Shot 기준 ↔ 원본 RH/LH 행 기준
재학습 없이 기존 예측을 같은 Shot의 양쪽 행에 붙여 평가한다.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
raw=pd.read_csv(root/'data/original_split/moldset_labeled_rg3.csv',parse_dates=['TimeStamp'])
p=pd.read_csv(t/'rg3_153_gas_best_shortshot_cascade_predictions.csv',parse_dates=['TimeStamp'])
assert len(raw)==1182 and len(p)==591 and p.TimeStamp.is_unique
raw['Reason']=raw.Reason.fillna('정상');raw['side']=raw.PART_NAME.str.extract('(LH|RH)$',expand=False)
assert not raw[['EQUIP_CD','TimeStamp','side']].duplicated().any()
assert raw.EQUIP_CD.nunique()==1
z=raw.merge(p,on='TimeStamp',how='left',validate='many_to_one',indicator=True)
assert z['_merge'].eq('both').all() and len(z)==1182
assert raw.groupby('TimeStamp').size().eq(2).all()
assert z.loc[z.PassOrFail.eq(1),'side'].eq('RH').all()
rows=[]
def metric(task,unit,truth,pred):
 tn,fp,fn,tp=confusion_matrix(np.asarray(truth,int),np.asarray(pred,int),labels=[0,1]).ravel()
 return dict(task=task,unit=unit,N=len(truth),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn) if tp+fn else 0,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
for task,types,reason,col in [
 ('gas_only',['정상','가스'],'가스','gas_prediction'),
 ('shortshot_only',['정상','미성형'],'미성형','shortshot_prediction'),
 ('cascade_any_defect',['정상','가스','미성형'],None,'final_defect')]:
 s=p[p.type.isin(types)]
 q=z[z.type.isin(types)]
 sy=s.type.ne('정상') if reason is None else s.type.eq(reason)
 ry=q.PassOrFail.eq(1) if reason is None else q.Reason.eq(reason)
 rows.append(metric(task,'Shot',sy,s[col]))
 rows.append(metric(task,'RH_LH_rows_same_prediction',ry,q[col]))
 rh=q.side.eq('RH')
 rows.append(metric(task,'RH_rows_only',ry[rh],q.loc[rh,col]))
 # Do NOT set LH=normal or tune a cavity policy using these evaluation labels.
m=pd.DataFrame(rows)
m.to_csv(t/'rg3_shot_vs_row_metrics.csv',index=False,encoding='utf-8-sig')
z.drop(columns='_merge').to_csv(t/'rg3_row_level_predictions.csv',index=False,encoding='utf-8-sig')
report='# RG3: 기존 Shot 예측을 원본 RH/LH 행 정답으로 재평가\n\n'
report+='재학습·튜닝 없음. 같은 Timestamp의 Shot 예측을 LH/RH 두 행에 그대로 붙인 결과. 같은 설비의 유일한 제품 RG3이며591Shot/1182행. 정상1132행, 가스17행, 미성형8행; 모든 불량은 이 데이터에서는RH에 기록됨.\n\n'
report+='가스/미성형 단독 비교는 Shot 평가와 동일하게 다른 유형의 불량Shot 양쪽 행을 제외해 범위를 맞춘다. 정상 캐비티 행은 해당Shot이 가스/미성형이면 그대로 정상 정답으로 평가하므로 행 기준 오탐에 포함된다. 최종전체 평가는1182행 모두.\n\n'
report+=m.to_csv(index=False)
report+='\n\n불량Shot을 맞히면 불량인RH는TP지만 정상LH는FP가 된다. 정상Shot 오탐은두 행에서FP가 된다. 불량이 있는Shot에라도알람이뜨면맞혔다고간주하는Shot성과와, 각캐비티 행 정답을맞혀야하는행성과는 목적이 다르다. RH행만평가하면현재자료에서는Shot성과와동일하지만공식전체행평가를RH만으로바꿀수있다는뜻은아니다. LH항상정상규칙은이번에적용하지않았다.\n\n'
report+='재현해놓은개발구간제외예측을평가단위만바꾼것이며새독립테스트아님. 학습/평가분할은여전히같은Shot의두행을동일구간에묶어야한다.\n'
(root/'reports/rg3_shot_vs_row_comparison.md').write_text(report,encoding='utf-8')
print(m.to_string(index=False))
print('Actual defective cavity rows:',raw[raw.PassOrFail.eq(1)].groupby(['side','Reason']).size().to_string())


              task                       unit    N  TP  FP  FN   TN  precision   recall       F1
          gas_only                       Shot  583   8   6   9  560   0.571429 0.470588 0.516129
          gas_only RH_LH_rows_same_prediction 1166   8  20   9 1129   0.285714 0.470588 0.355556
          gas_only               RH_rows_only  583   8   6   9  560   0.571429 0.470588 0.516129
    shortshot_only                       Shot  574   6  11   2  555   0.352941 0.750000 0.480000
    shortshot_only RH_LH_rows_same_prediction 1148   6  28   2 1112   0.176471 0.750000 0.285714
    shortshot_only               RH_rows_only  574   6  11   2  555   0.352941 0.750000 0.480000
cascade_any_defect                       Shot  591  14  17  11  549   0.451613 0.560000 0.500000
cascade_any_defect RH_LH_rows_same_prediction 1182  14  48  11 1109   0.225806 0.560000 0.321839
cascade_any_defect               RH_rows_only  591  14  17  11  549   0.451613 0.560000 0.500000
Actual defective cavity rows: 